In [1]:
using Symbolics#, OffsetArrays, Nemo
using SymbolicUtils
using LinearAlgebra

In [3]:
# @variables λ μ Δx Δy ux_i_jp05 ux_i_jm05 uy_i_jp05 uy_i_jm05 ux_ip05_j ux_im05_j uy_ip05_j uy_im05_j

In [4]:
# eq = λ*Δx*ux_i_jp05 + (λ+2*μ)*Δx*uy_i_jp05 - λ*Δx*ux_i_jm05 - (λ+2*μ)*Δx*uy_i_jm05 + (λ+2*μ)*Δy*ux_ip05_j + λ*Δy*uy_ip05_j - (λ+2*μ)*Δy*ux_im05_j - λ*Δy*uy_im05_j

In [5]:
# dict = Dict(
#     ux_i_jp05 => ,
#     ux_i_jm05 => ,
#     uy_i_jp05 => ,
#     uy_i_jm05 => ,
#     ux_ip05_j => ,
#     ux_im05_j => ,
#     uy_ip05_j => ,
#     uy_im05_j => ,
# )

In [51]:
@variables a b c Δ

δ = (a - c)/2

λ1 = (a + c)/2 + Δ
λ2 = (a + c)/2 - Δ

v1 = [b/sqrt(b^2+(λ1-a)^2),
      (λ1-a)/sqrt(b^2+(λ1-a)^2)]

v2 = [b/sqrt(b^2+(λ2-a)^2),
      (λ2-a)/sqrt(b^2+(λ2-a)^2)]

Q = [v1[1] v2[1];
     v1[2] v2[2]]

QT = transpose(Q)

test = simplify.(expand.(QT*Q))

# Now enforce Δ^2 identity
test = substitute(test, Dict(Δ^2 => δ^2 + b^2))

simplify.(test)

2×2 Matrix{Num}:
 1  0
 0  1

In [59]:
Λ = [λ1 0; 0 λ2]

test2 = Q * Λ * QT

test2 = simplify.(expand.(test2))

# Now enforce Δ^2 identity
test2 = substitute(test2, Dict(Δ^2 => δ^2 + b^2))

simplify.(expand.(test2))

2×2 Matrix{Num}:
                                             ((1//2)*(a^3 + c^3)*(b^2) - (1//2)*(a^2)*(b^2)*c + (2//1)*(a + c)*(b^4) - (1//2)*a*(b^2)*(c^2) + (2//1)*a*(b^2)*(Δ^2) - (2//1)*(b^2)*c*(Δ^2)) / ((1//4)*(a^4 + c^4) - (a^3)*c + (2//1)*(a^2 + c^2)*(b^2) + (3//2)*(a^2)*(c^2) - (a^2 + c^2)*(Δ^2) - (4//1)*a*(b^2)*c - a*(c^3) + (2//1)*a*c*(Δ^2) + 4(b^4))  …                                                                                                                                             (-(1//2)*(a^3)*b*c + (a^2)*(b^3) + (3//2)*(a^2)*b*(c^2) - (4//1)*a*(b^3)*c - (3//2)*a*b*(c^3) + (2//1)*a*b*c*(Δ^2) + 4(b^5) + (3//1)*(b^3)*(c^2) + (1//2)*b*(c^4) - (2//1)*b*(c^2)*(Δ^2)) / ((1//4)*(a^4 + c^4) - (a^3)*c + (2//1)*(a^2 + c^2)*(b^2) + (3//2)*(a^2)*(c^2) - (a^2 + c^2)*(Δ^2) - (4//1)*a*(b^2)*c - a*(c^3) + (2//1)*a*c*(Δ^2) + 4(b^4))
 (-(1//2)*(a^3)*b*c + (a^2)*(b^3) + (3//2)*(a^2)*b*(c^2) - (4//1)*a*(b^3)*c - (3//2)*a*b*(c^3) + (2//1)*a*b*c*(Δ^2) + 4(b^5) + (3//1)*(b^3)*(c^2) + (1/

In [62]:
Λ = [λ1 0; 0 λ2]

test2 = Q * Λ * transpose(Q)

# Expand fully first
test2 = expand.(test2)

# Substitute WITHOUT Ref and WITHOUT broadcasting
test2 = substitute(test2, Dict(Δ^2 => δ^2 + b^2))

# Expand again
test2 = expand.(test2)

# Final simplification
simplify.(test2; expand=true)

# Verify against A
simplify.(expand.(test2 .- A))

2×2 Matrix{Num}:
                             (-a*((1//2)*(a^2 + c^2) - a*(c + Δ) + 2(b^2) + c*Δ)*((1//2)*(a^2 + c^2) - a*c + a*Δ + 2(b^2) - c*Δ) + ((1//2)*(a + c)*(b^2) + (b^2)*Δ)*((1//2)*(a^2 + c^2) - a*c + a*Δ + 2(b^2) - c*Δ) + ((1//2)*(a + c)*(b^2) - (b^2)*Δ)*((1//2)*(a^2 + c^2) - a*(c + Δ) + 2(b^2) + c*Δ)) / (((1//2)*(a^2 + c^2) - a*(c + Δ) + 2(b^2) + c*Δ)*((1//2)*(a^2 + c^2) - a*c + a*Δ + 2(b^2) - c*Δ))  …                                                                                                                                                                           (-((1//2)*(a^2 + c^2) - a*(c + Δ) + 2(b^2) + c*Δ)*((1//2)*(a^2 + c^2) - a*c + a*Δ + 2(b^2) - c*Δ)*b + (-(1//2)*a*b*c + b^3 + (1//2)*b*(c^2) + b*c*Δ)*((1//2)*(a^2 + c^2) - a*c + a*Δ + 2(b^2) - c*Δ) + (-(1//2)*a*b*c + b^3 + (1//2)*b*(c^2) - b*c*Δ)*((1//2)*(a^2 + c^2) - a*(c + Δ) + 2(b^2) + c*Δ)) / (((1//2)*(a^2 + c^2) - a*(c + Δ) + 2(b^2) + c*Δ)*((1//2)*(a^2 + c^2) - a*c + a*Δ + 2(b^2) - c*Δ))
 (-((1//2)*(a^2 + c

In [61]:
R = expand(Q * Λ * transpose(Q) - A)
simplify.(R; expand=true)

2×2 Matrix{Num}:
                                        (-a*((1//4)*(a^2 + c^2) - (1//2)*a*c - a*Δ + b^2 + c*Δ + Δ^2)*((1//4)*(a^2 + c^2) - (1//2)*a*c + a*Δ + b^2 - c*Δ + Δ^2) + ((1//2)*(a + c)*(b^2) + (b^2)*Δ)*((1//4)*(a^2 + c^2) - (1//2)*a*c + a*Δ + b^2 - c*Δ + Δ^2) + ((1//2)*(a + c)*(b^2) - (b^2)*Δ)*((1//4)*(a^2 + c^2) - (1//2)*a*c - a*Δ + b^2 + c*Δ + Δ^2)) / (((1//4)*(a^2 + c^2) - (1//2)*a*c - a*Δ + b^2 + c*Δ + Δ^2)*((1//4)*(a^2 + c^2) - (1//2)*a*c + a*Δ + b^2 - c*Δ + Δ^2))  …                                                                                                                                                                                          (((1//4)*(a^2 + c^2) - (1//2)*a*c - a*Δ + b^2 + c*Δ + Δ^2)*(-(1//4)*(a^2)*b + (1//4)*b*(c^2) - b*c*Δ + b*(Δ^2)) + ((1//4)*(a^2 + c^2) - (1//2)*a*c + a*Δ + b^2 - c*Δ + Δ^2)*(-(1//4)*(a^2)*b + (1//4)*b*(c^2) + b*c*Δ + b*(Δ^2)) - ((1//4)*(a^2 + c^2) - (1//2)*a*c - a*Δ + b^2 + c*Δ + Δ^2)*((1//4)*(a^2 + c^2) - (1//2)*a*c + a*Δ + b^